# Tuning the model

## Leraning Rate

In [1]:
from src.cnn_utils import get_dataset

train_dataset, val_dataset = get_dataset()

# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

Training dataset size: 24000
Validation dataset size: 6000


In [2]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import wandb


# =========================================================
# MODEL
# =========================================================
# A simple convolutional block: Conv2d -> BatchNorm -> ReLU -> (optional MaxPool)

class ConvBlock(nn.Module):
    """Conv2d -> BatchNorm2d -> ReLU -> optional MaxPool2d"""
    def __init__(self, in_channels, out_channels, use_pool=False):
        super().__init__()

        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]

        if use_pool:
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

# constructing CNN with depth 12 from previous runs
class DepthCNN(nn.Module):
    def __init__(
        self,
        depth=12,
        in_channels=3,
        num_classes=10,
        base_channels=32,
        max_channels=256,
        dropout=0.5,
        inputsize=224,
    ):
        super().__init__()

        layers = []
        current_in = in_channels
        current_out = base_channels
        current_size = inputsize
        pool_count = 0

        for i in range(depth):
            want_pool = ((i + 1) % 2 == 0)
            use_pool = want_pool and current_size >= 2 and pool_count < 4

            layers.append(ConvBlock(current_in, current_out, use_pool))

            current_in = current_out

            if use_pool:
                current_size //= 2
                pool_count += 1
                current_out = min(current_out * 2, max_channels)

        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(current_in, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# =========================================================
# TRAIN / EVAL
# =========================================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    start_time = time.time()

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    epoch_time = time.time() - start_time

    return epoch_loss, epoch_acc, epoch_time


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total

    return epoch_loss, epoch_acc


# =========================================================
# SETTINGS
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

depth = 12 # depth evaluated from previous runs
epochs = 50 # since in previous runs the model converged at around 30 epochs, we can reduce the number of epochs for tuning to save time
batch_size = 64 # default value
weight_decay = 1e-4 # default value

# try different learning rates (some values are absurd)
lrs = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True
)


# =========================================================
# LR TUNING LOOP
# =========================================================

results = {}

for lr in lrs:
    print("\n" + "=" * 80)
    print(f"Training with lr = {lr}")
    print("=" * 80)

    model = DepthCNN(depth=depth).to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        nesterov=True,
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_val_loss = float("inf")
    best_train_loss = float("inf")
    best_epoch = -1

    run = wandb.init(
        project="MPW-CNN",
        entity="MSE_DeLearn_SPR26",
        name=f"lr_tuning_MomentumNesterov_depth12_lr{lr}",
        config={
            "depth": depth,
            "lr": lr,
            "optimizer": "SGD_Nesterov",
            "momentum": 0.9,
            "weight_decay": weight_decay,
            "epochs": epochs,
            "batch_size": batch_size,
        },
        reinit=True
    )

    for epoch in range(epochs):
        train_loss, train_acc, epoch_time = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1

        if train_acc > best_train_acc:
            best_train_acc = train_acc

        if val_loss < best_val_loss:
            best_val_loss = val_loss

        if train_loss < best_train_loss:
            best_train_loss = train_loss

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
            f"time={epoch_time:.1f}s"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "best_val_acc": best_val_acc,
            "best_train_acc": best_train_acc,
            "best_val_loss": best_val_loss,
            "best_train_loss": best_train_loss,
            "epoch_time_sec": epoch_time,
        })

    wandb.summary["best_val_acc"] = best_val_acc
    wandb.summary["best_train_acc"] = best_train_acc
    wandb.summary["best_val_loss"] = best_val_loss
    wandb.summary["best_train_loss"] = best_train_loss
    wandb.summary["best_epoch"] = best_epoch
    wandb.finish()

    results[lr] = {
        "lr": lr,
        "best_val_acc": best_val_acc,
        "best_train_acc": best_train_acc,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_train_loss": best_train_loss,
    }


# =========================================================
# SUMMARY
# =========================================================

print("\n" + "=" * 80)
print("LR RESULTS")
print("=" * 80)

for lr, result in results.items():
    print(
        f"lr={lr:<8} | "
        f"best_val_acc={result['best_val_acc']:.4f} | "
        f"best_train_acc={result['best_train_acc']:.4f} | "
        f"best_epoch={result['best_epoch']} | "
        f"best_val_loss={result['best_val_loss']:.4f} | "
        f"best_train_loss={result['best_train_loss']:.4f}"
    )

best_lr = max(results, key=lambda lr: results[lr]["best_val_acc"])

print("\nBest LR:")
print(f"{best_lr} with val_acc={results[best_lr]['best_val_acc']:.4f}")


# =========================================================
# W&B COMPARISON TABLE
# =========================================================

if wandb.run is not None:
    wandb.finish()

run = wandb.init(
    project="MPW-CNN",
    entity="MSE_DeLearn_SPR26",
    name="lr_tuning_MomentumNesterov_summary_depth12",
    reinit=True
)

comparison_table = wandb.Table(columns=[
    "lr",
    "best_val_acc",
    "best_train_acc",
    "best_epoch",
    "best_val_loss",
    "best_train_loss",
])

for lr, result in results.items():
    comparison_table.add_data(
        result["lr"],
        result["best_val_acc"],
        result["best_train_acc"],
        result["best_epoch"],
        result["best_val_loss"],
        result["best_train_loss"],
    )

wandb.log({
    "lr_comparison_table": comparison_table,
})

best_lr = max(results, key=lambda lr: results[lr]["best_val_acc"])

wandb.summary["best_lr"] = best_lr
wandb.summary["best_val_acc"] = results[best_lr]["best_val_acc"]
wandb.summary["best_train_acc"] = results[best_lr]["best_train_acc"]
wandb.summary["best_epoch"] = results[best_lr]["best_epoch"]
wandb.summary["best_val_loss"] = results[best_lr]["best_val_loss"]
wandb.summary["best_train_loss"] = results[best_lr]["best_train_loss"]

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lucas\_netrc.



Training with lr = 0.1


wandb: Currently logged in as: lucas-j-keller98 (MSE_DeLearn_SPR26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 01/50 | train_loss=2.0293 | train_acc=0.2399 | val_loss=1.9236 | val_acc=0.2988 | time=49.5s
Epoch 02/50 | train_loss=1.6928 | train_acc=0.3867 | val_loss=1.7095 | val_acc=0.4192 | time=49.0s
Epoch 03/50 | train_loss=1.4089 | train_acc=0.5035 | val_loss=1.8362 | val_acc=0.4012 | time=49.3s
Epoch 04/50 | train_loss=1.2251 | train_acc=0.5697 | val_loss=1.4327 | val_acc=0.5032 | time=50.6s
Epoch 05/50 | train_loss=1.0937 | train_acc=0.6160 | val_loss=1.1506 | val_acc=0.5943 | time=50.4s
Epoch 06/50 | train_loss=0.9653 | train_acc=0.6680 | val_loss=0.9490 | val_acc=0.6692 | time=49.3s
Epoch 07/50 | train_loss=0.8587 | train_acc=0.7109 | val_loss=0.9170 | val_acc=0.6828 | time=49.4s
Epoch 08/50 | train_loss=0.7581 | train_acc=0.7430 | val_loss=0.9904 | val_acc=0.6827 | time=49.4s
Epoch 09/50 | train_loss=0.6775 | train_acc=0.7719 | val_loss=0.7934 | val_acc=0.7328 | time=49.3s
Epoch 10/50 | train_loss=0.5922 | train_acc=0.7998 | val_loss=0.7652 | val_acc=0.7462 | time=49.4s
Epoch 11/5

best_train_acc,▁▂▃▄▅▆▆▆▇▇▇▇▇▇▇█████████████████████████
best_train_loss,█▇▆▅▅▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,▁▃▃▄▅▆▆▇▇▇▇▇████████████████████████████
best_val_loss,█▇▇▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
epoch_time_sec,▃▁▂█▇▂▃▂▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃▂▂▂▂▂▂▃▂▃▂▂▂
train_acc,▁▂▄▅▅▆▆▆▇▇▇▇▇▇▇█████████████████████████
train_loss,█▇▆▅▅▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▃▂▄▅▆▆▇▇▇▇▆█▇▇▇▇█▇▇█▇███▆██▇▇███▇█▇▇█▇▇
val_loss,█▇█▄▃▃▂▂▂▅▃▂▂▁▄▁▄▃▂▂▂▁▁▃▇▁▃▂▃▇▂▁▄▁▃▅▁▅▄▂
best_epoch,46


...


Training with lr = 1e-05


Epoch 01/50 | train_loss=2.3304 | train_acc=0.1134 | val_loss=2.2569 | val_acc=0.1538 | time=49.3s
Epoch 02/50 | train_loss=2.2572 | train_acc=0.1555 | val_loss=2.1964 | val_acc=0.2002 | time=49.1s
Epoch 03/50 | train_loss=2.2074 | train_acc=0.1881 | val_loss=2.1630 | val_acc=0.2343 | time=49.1s
Epoch 04/50 | train_loss=2.1787 | train_acc=0.2005 | val_loss=2.1347 | val_acc=0.2640 | time=49.2s
Epoch 05/50 | train_loss=2.1534 | train_acc=0.2205 | val_loss=2.1110 | val_acc=0.2842 | time=49.2s
Epoch 06/50 | train_loss=2.1283 | train_acc=0.2312 | val_loss=2.0912 | val_acc=0.2927 | time=49.2s
Epoch 07/50 | train_loss=2.1081 | train_acc=0.2410 | val_loss=2.0754 | val_acc=0.2952 | time=49.2s
Epoch 08/50 | train_loss=2.0903 | train_acc=0.2507 | val_loss=2.0539 | val_acc=0.2998 | time=49.2s
Epoch 09/50 | train_loss=2.0684 | train_acc=0.2607 | val_loss=2.0361 | val_acc=0.3040 | time=49.2s
Epoch 10/50 | train_loss=2.0565 | train_acc=0.2672 | val_loss=2.0197 | val_acc=0.3115 | time=49.2s
Epoch 11/5

best_train_acc,▁▂▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████
best_train_loss,█▇▇▇▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
best_val_acc,▁▂▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█████
best_val_loss,█▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
epoch_time_sec,▄▁▁▂▂▃▂▂▃▂▂▂▂▂▂▁▂▂▂▂▂▂▂▆█▃▂▁▂▂▂▂▂▂▁▁▂▁▂▂
train_acc,▁▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████
train_loss,█▇▇▇▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁
val_acc,▁▂▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████
val_loss,█▇▇▇▇▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
best_epoch,50



LR RESULTS
lr=0.1      | best_val_acc=0.8462 | best_train_acc=0.9803 | best_epoch=46 | best_val_loss=0.5507 | best_train_loss=0.0652
lr=0.01     | best_val_acc=0.9042 | best_train_acc=1.0000 | best_epoch=48 | best_val_loss=0.3773 | best_train_loss=0.0006
lr=0.001    | best_val_acc=0.8753 | best_train_acc=1.0000 | best_epoch=46 | best_val_loss=0.4200 | best_train_loss=0.0023
lr=0.0001   | best_val_acc=0.7097 | best_train_acc=0.9643 | best_epoch=50 | best_val_loss=0.8772 | best_train_loss=0.1953
lr=1e-05    | best_val_acc=0.4817 | best_train_acc=0.4590 | best_epoch=50 | best_val_loss=1.5367 | best_train_loss=1.5770

Best LR:
0.01 with val_acc=0.9042


best_epoch,48
best_lr,0.01
best_train_acc,1
best_train_loss,0.00056
best_val_acc,0.90417
best_val_loss,0.37735


## Playing with the momentum

In [4]:
from src.cnn_utils import get_dataset

train_dataset, val_dataset = get_dataset()

# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

Training dataset size: 24000
Validation dataset size: 6000


In [6]:
import time
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import wandb
from src.cnn_utils import *

# ---------------------------------------------------------
# config
# ---------------------------------------------------------
model_cfg = ModelConfig(depth=12)
wandb_cfg = WandbConfig(use_wandb=True)

device = get_device()

momentums = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0] # testing different momentum values from 0 (no momentum) to 1 (full momentum)
batch_size = 64 # default value
epochs = 50
lr = 0.01 # best learning rate from previous tuning
weight_decay = 1e-4 # default value

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

criterion = nn.CrossEntropyLoss()

# ---------------------------------------------------------
# momentum sweep
# ---------------------------------------------------------

all_results = {} # to store results for each momentum value

wandb_login_if_needed(wandb_cfg)

for m in momentums:
    print(f"\n==== Training with momentum = {m} ====\n")

    # new model and optimizer for each momentum value
    model = build_model(model_cfg).to(device)
    num_parameters = get_num_parameters(model)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=m,
        weight_decay=weight_decay,
    )

    run = init_wandb_run(
        cfg=wandb_cfg,
        model=model,
        run_name=f"Momentum_{m}_depth{model_cfg.depth}_lr{lr}_e{epochs}",
        config_dict={
            "optimizer": "Momentum_SGD",
            "momentum": m,
            "lr": lr,
            "weight_decay": weight_decay,
            "depth": model_cfg.depth,
            "batch_size": batch_size,
            "epochs": epochs,
            "num_parameters": num_parameters,
            "device": str(device),
        },
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "epoch_time_sec": [],
    }

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_epoch = -1
    best_train_acc = 0.0
    best_train_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())

    # ---------------------------
    # training loop
    # ---------------------------
    for epoch in range(epochs):

        # TRAIN
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        start_time = time.time()

        for x, y in train_loader:
            x, y = x.to(device,non_blocking=True), y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            batch_n = y.size(0)
            train_loss_sum += loss.item() * batch_n
            train_correct += (logits.argmax(dim=1) == y).sum().item()
            train_total += batch_n

        train_loss = train_loss_sum / train_total
        train_acc = train_correct / train_total

        # VALIDATION
        model.eval()
        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

                logits = model(x)
                loss = criterion(logits, y)

                batch_n = y.size(0)
                val_loss_sum += loss.item() * batch_n
                val_correct += (logits.argmax(dim=1) == y).sum().item()
                val_total += batch_n

        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        epoch_time = time.time() - start_time

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["epoch_time_sec"].append(epoch_time)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_train_acc = train_acc
            best_train_loss = train_loss
            best_model_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch [{epoch+1:02d}/{epochs:02d}] | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
            f"time={epoch_time:.1f}s"
        )

        if run is not None:
            wandb.log({
                "epoch": epoch + 1,
                "train/loss": train_loss,
                "train/accuracy": train_acc,
                "val/loss": val_loss,
                "val/accuracy": val_acc,
                "epoch_time_sec": epoch_time,
                "momentum": m,
                "best_val_accuracy_so_far": best_val_acc,
            })

    # restore best model state for this momentum run
    model.load_state_dict(best_model_state)

    avg_epoch_time = sum(history["epoch_time_sec"]) / len(history["epoch_time_sec"])

    all_results[m] = {
        "momentum": m,
        "depth": model_cfg.depth,
        "num_parameters": num_parameters,
        "best_val_acc": best_val_acc,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "best_train_acc": best_train_acc,
        "best_train_loss": best_train_loss,
        "final_train_acc": history["train_acc"][-1],
        "final_val_acc": history["val_acc"][-1],
        "avg_epoch_time_sec": avg_epoch_time,
    }

    if run is not None:
        wandb.summary["best_epoch"] = best_epoch
        wandb.summary["best_val_accuracy"] = best_val_acc
        wandb.summary["best_val_loss"] = best_val_loss
        wandb.summary["best_train_accuracy_at_best_val"] = best_train_acc
        wandb.summary["best_train_loss_at_best_val"] = best_train_loss
        wandb.summary["final_train_accuracy"] = history["train_acc"][-1]
        wandb.summary["final_val_accuracy"] = history["val_acc"][-1]
        wandb.summary["avg_epoch_time_sec"] = avg_epoch_time
        wandb.finish()

# =========================================================
# OPTIONAL: W&B SUMMARY RUN (MOMENTUM COMPARISON)
# =========================================================

if wandb_cfg.use_wandb and wandb_cfg.mode != "disabled":
    best_momentum = max(all_results.keys(), key=lambda k: all_results[k]["best_val_acc"])

    run = wandb.init(
        project=wandb_cfg.project,
        entity=wandb_cfg.entity,
        mode=wandb_cfg.mode,
        name=f"momentum_comparison_depth{model_cfg.depth}",
        reinit=True,
        settings=wandb.Settings(init_timeout=wandb_cfg.init_timeout),
    )

    comparison_table = wandb.Table(columns=[
        "momentum",
        "depth",
        "num_parameters",
        "best_val_acc",
        "best_val_loss",
        "best_epoch",
        "best_train_acc",
        "best_train_loss",
        "final_train_acc",
        "final_val_acc",
        "avg_epoch_time_sec",
    ])

    for momentum_value, result in all_results.items():
        comparison_table.add_data(
            result["momentum"],
            result["depth"],
            result["num_parameters"],
            result["best_val_acc"],
            result["best_val_loss"],
            result["best_epoch"],
            result["best_train_acc"],
            result["best_train_loss"],
            result["final_train_acc"],
            result["final_val_acc"],
            result["avg_epoch_time_sec"],
        )

    wandb.log({
        "momentum_comparison_table": comparison_table,
        "best_momentum": best_momentum,
        "best_momentum_val_acc": all_results[best_momentum]["best_val_acc"],
        "depth": model_cfg.depth,
    })

    wandb.summary["best_momentum"] = best_momentum
    wandb.summary["best_momentum_val_acc"] = all_results[best_momentum]["best_val_acc"]
    wandb.summary["depth"] = model_cfg.depth

    wandb.finish()


# ---------------------------------------------------------
# optional console summary
# ---------------------------------------------------------
print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)

for m, result in all_results.items():
    print(
        f"Momentum={result['momentum']:.1f} | "
        f"depth={result['depth']} | "
        f"params={result['num_parameters']:,} | "
        f"best_val_acc={result['best_val_acc']:.4f} | "
        f"best_val_loss={result['best_val_loss']:.4f} | "
        f"best_epoch={result['best_epoch']} | "
        f"final_train_acc={result['final_train_acc']:.4f} | "
        f"final_val_acc={result['final_val_acc']:.4f} | "
        f"avg_epoch_time={result['avg_epoch_time_sec']:.2f}s"
    )

best_momentum = max(all_results.keys(), key=lambda k: all_results[k]["best_val_acc"])
print("\nBest momentum based on validation accuracy:")
print(f"Momentum = {best_momentum}")
print(f"Best validation accuracy = {all_results[best_momentum]['best_val_acc']:.4f}")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.



==== Training with momentum = 0.0 ====



Epoch [01/50] | train_loss=1.7805 | train_acc=0.3726 | val_loss=1.5002 | val_acc=0.4737 | time=72.1s
Epoch [02/50] | train_loss=1.3840 | train_acc=0.5223 | val_loss=2.0908 | val_acc=0.3685 | time=54.5s
Epoch [03/50] | train_loss=1.1888 | train_acc=0.5932 | val_loss=2.5703 | val_acc=0.3217 | time=54.7s
Epoch [04/50] | train_loss=1.0468 | train_acc=0.6424 | val_loss=1.1278 | val_acc=0.6027 | time=54.4s
Epoch [05/50] | train_loss=0.9308 | train_acc=0.6847 | val_loss=1.4881 | val_acc=0.5095 | time=54.4s
Epoch [06/50] | train_loss=0.8375 | train_acc=0.7170 | val_loss=1.0087 | val_acc=0.6568 | time=54.7s
Epoch [07/50] | train_loss=0.7567 | train_acc=0.7481 | val_loss=1.1879 | val_acc=0.5880 | time=54.7s
Epoch [08/50] | train_loss=0.6885 | train_acc=0.7710 | val_loss=1.2071 | val_acc=0.5923 | time=54.7s
Epoch [09/50] | train_loss=0.6193 | train_acc=0.7957 | val_loss=1.2928 | val_acc=0.6053 | time=54.7s
Epoch [10/50] | train_loss=0.5516 | train_acc=0.8205 | val_loss=0.8110 | val_acc=0.7320 | t

best_val_accuracy_so_far,▁▁▁▃▃▄▄▄▆▆▆▆▆▆▆▆▆▆▇▇▇▇██████████████████
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
epoch_time_sec,█▂▂▂▂▂▂▂▂▁▁▁▁▂▂▂▂▂▁▁▁▁▂▂▂▁▁▁▁▁▂▂▂▂▁▁▂▁▂▁
momentum,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/accuracy,▁▃▃▄▄▅▆▆▆▆▇▇▇▇▇█████████████████████████
train/loss,█▆▆▅▅▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/accuracy,▃▂▁▅▃▄▄▅▆▆▃▆▄▃▇▇▅▆▇▇▄▇███▇███▇██████████
val/loss,▃▅▃▃▂▃▃▂▂▂▂▂▅█▂▂▃▃▁▂▅▂▁▁▁▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁
avg_epoch_time_sec,54.12416
best_epoch,50
best_train_accuracy_at_best_val,1



==== Training with momentum = 0.2 ====



Epoch [01/50] | train_loss=1.7695 | train_acc=0.3736 | val_loss=1.5538 | val_acc=0.4498 | time=54.1s
Epoch [02/50] | train_loss=1.3655 | train_acc=0.5265 | val_loss=1.3827 | val_acc=0.5145 | time=53.0s
Epoch [03/50] | train_loss=1.1659 | train_acc=0.5995 | val_loss=1.3366 | val_acc=0.5543 | time=53.1s
Epoch [04/50] | train_loss=1.0363 | train_acc=0.6473 | val_loss=1.2045 | val_acc=0.5842 | time=53.0s
Epoch [05/50] | train_loss=0.9165 | train_acc=0.6887 | val_loss=1.0948 | val_acc=0.6053 | time=53.0s
Epoch [06/50] | train_loss=0.8140 | train_acc=0.7253 | val_loss=0.8867 | val_acc=0.7017 | time=53.1s
Epoch [07/50] | train_loss=0.7287 | train_acc=0.7560 | val_loss=1.6805 | val_acc=0.5328 | time=52.9s
Epoch [08/50] | train_loss=0.6535 | train_acc=0.7843 | val_loss=1.0096 | val_acc=0.6613 | time=54.0s
Epoch [09/50] | train_loss=0.5795 | train_acc=0.8065 | val_loss=0.8691 | val_acc=0.7120 | time=54.5s
Epoch [10/50] | train_loss=0.5250 | train_acc=0.8286 | val_loss=1.2223 | val_acc=0.6438 | t

KeyboardInterrupt: 

## Playing with the batchsize